In [1]:
from pathlib import Path
import sys
from time import perf_counter

import torch


# ============================================================
# 1. Locate the repository and make src/ importable.
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print("Project root:", PROJECT_ROOT)


# ============================================================
# 2. Fixed paths and generation settings.
# ============================================================

DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
)

TOKEN_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/final_model/tokens"
)

FORECASTING_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "forecasting.yaml"
)

DYNAMIC_GRAPH_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "dynamic_graph.yaml"
)

TRAIN_CACHE_PATH = (
    TOKEN_DIR
    / "origin_aligned_train_tokens.pt"
)

VAL_CACHE_PATH = (
    TOKEN_DIR
    / "origin_aligned_val_tokens.pt"
)

SERIES_BATCH_SIZE = 93
WINDOW_BATCH_SIZE = 2
OVERWRITE = False

TOKEN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for required_path in (
    DATA_DIR,
    FORECASTING_CONFIG_PATH,
    DYNAMIC_GRAPH_CONFIG_PATH,
):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path does not exist: {required_path}"
        )


# ============================================================
# 3. Repository imports.
# ============================================================

from src.data.load_candle_data import (
    clean_candle_splits,
    load_candle_splits,
)

from src.data.data_generator import (
    WindowedCandleDataset,
)

from src.data.token_graph_dataset import (
    build_and_save_origin_aligned_token_cache,
    load_origin_aligned_token_cache,
)

from src.models.dynamic_graph.config import (
    build_dense_window_config,
    load_dynamic_graph_config,
)

from src.models.kronos_tokenizer import (
    KronosTokenizerAdapter,
)

from src.utils.config import (
    load_yaml,
)


# ============================================================
# 4. Load configuration, real data and frozen tokenizer.
# ============================================================

forecasting_config = load_yaml(
    FORECASTING_CONFIG_PATH
)

experiment_config = load_dynamic_graph_config(
    DYNAMIC_GRAPH_CONFIG_PATH,
    preset="structured_parallel_uniform",
)

train_raw, val_raw, test_raw = load_candle_splits(
    DATA_DIR
)

train, val, test = clean_candle_splits(
    train_raw,
    val_raw,
    test_raw,
)

kronos_tokenizer = (
    KronosTokenizerAdapter.from_config(
        forecasting_config,
        series_batch_size=SERIES_BATCH_SIZE,
    )
    .load()
)

print(
    "Tokenizer loaded:",
    kronos_tokenizer.tokenizer_id,
)


# ============================================================
# 5. Build the model-specific dense 1...60 window config.
#
# The baseline forecasting configuration is copied and the
# final model's target horizons are overridden to 1...60.
# ============================================================

dense_window_config = build_dense_window_config(
    forecasting_config,
    experiment_config,
)

dense_train_dataset = WindowedCandleDataset.from_config(
    split=train,
    config=dense_window_config,
    normaliser=None,
)

dense_val_dataset = WindowedCandleDataset.from_config(
    split=val,
    config=dense_window_config,
    normaliser=None,
)


# ============================================================
# 6. Validate the window contract before starting encoding.
# ============================================================

expected_horizons = tuple(
    range(1, 61)
)

if tuple(dense_train_dataset.horizons) != expected_horizons:
    raise RuntimeError(
        "Training dataset does not use dense horizons 1...60."
    )

if tuple(dense_val_dataset.horizons) != expected_horizons:
    raise RuntimeError(
        "Validation dataset does not use dense horizons 1...60."
    )

if dense_train_dataset.context_length != 60:
    raise RuntimeError(
        "Training context length is not 60."
    )

if dense_val_dataset.context_length != 60:
    raise RuntimeError(
        "Validation context length is not 60."
    )

print()
print(
    "Training sessions:",
    len(train["samples"]),
)

print(
    "Validation sessions:",
    len(val["samples"]),
)

print(
    "Training windows:",
    len(dense_train_dataset),
)

print(
    "Validation windows:",
    len(dense_val_dataset),
)

print(
    "Dense horizons:",
    dense_train_dataset.horizons[:5],
    "...",
    dense_train_dataset.horizons[-1],
)


# ============================================================
# 7. Helper functions.
# ============================================================

def generate_or_load_cache(
    *,
    split_name: str,
    dataset: WindowedCandleDataset,
    output_path: Path,
):
    """Generate one cache, or load an existing validated cache."""
    if output_path.exists() and not OVERWRITE:
        print()
        print(
            f"{split_name} cache already exists; loading:",
            output_path,
        )

        cache = load_origin_aligned_token_cache(
            output_path
        )

        return cache, 0.0

    if output_path.exists() and OVERWRITE:
        output_path.unlink()

    print()
    print(
        f"Generating {split_name.lower()} cache:",
        output_path,
    )

    start_time = perf_counter()

    saved_path = build_and_save_origin_aligned_token_cache(
        dataset=dataset,
        tokenizer=kronos_tokenizer,
        path=output_path,
        evaluation_horizons=(
            1,
            5,
            15,
            30,
            60,
        ),
        window_batch_size=WINDOW_BATCH_SIZE,
        series_batch_size=SERIES_BATCH_SIZE,
        prefix_check_batches=1,
        show_progress=True,
    )

    duration_seconds = (
        perf_counter()
        - start_time
    )

    print()
    print(
        f"Saved {split_name.lower()} cache to:",
        saved_path,
    )

    print(
        f"{split_name} generation time:",
        f"{duration_seconds / 60.0:.2f} minutes",
    )

    cache = load_origin_aligned_token_cache(
        output_path
    )

    return cache, duration_seconds


def print_cache_summary(
    *,
    split_name: str,
    cache,
) -> None:
    print()
    print(f"{split_name} cache:")

    print(
        "  context:",
        tuple(
            cache["context_tokens"].shape
        ),
    )

    print(
        "  target s1:",
        tuple(
            cache["target_s1"].shape
        ),
    )

    print(
        "  target s2:",
        tuple(
            cache["target_s2"].shape
        ),
    )

    print(
        "  evaluation truth:",
        tuple(
            cache["evaluation_true"].shape
        ),
    )

    print(
        "  context mean:",
        tuple(
            cache["context_mean"].shape
        ),
    )

    print(
        "  context std:",
        tuple(
            cache["context_std"].shape
        ),
    )

    print(
        "  target indices:",
        tuple(
            cache["target_indices"].shape
        ),
    )

    print()
    print(
        f"{split_name} future clipping by channel:"
    )

    for channel, value in zip(
        cache["tokenizer_channels"],
        cache[
            "future_clipping_rate_percent_by_channel"
        ].tolist(),
    ):
        print(
            f"  {channel}: {value:.6f}%"
        )


# ============================================================
# 8. Generate VALIDATION first.
#
# This is the smaller split and confirms the complete generation,
# atomic-save and reload path before the longer training run.
# ============================================================

val_token_cache, val_duration_seconds = (
    generate_or_load_cache(
        split_name="Validation",
        dataset=dense_val_dataset,
        output_path=VAL_CACHE_PATH,
    )
)

print_cache_summary(
    split_name="Validation",
    cache=val_token_cache,
)


# ============================================================
# 9. Generate TRAINING after validation has passed.
# ============================================================

train_token_cache, train_duration_seconds = (
    generate_or_load_cache(
        split_name="Training",
        dataset=dense_train_dataset,
        output_path=TRAIN_CACHE_PATH,
    )
)

print_cache_summary(
    split_name="Training",
    cache=train_token_cache,
)


# ============================================================
# 10. Final alignment checks.
# ============================================================

if train_token_cache["asset_cols"] != val_token_cache["asset_cols"]:
    raise RuntimeError(
        "Training and validation asset ordering differs."
    )

if (
    tuple(train_token_cache["evaluation_horizons"])
    != tuple(val_token_cache["evaluation_horizons"])
):
    raise RuntimeError(
        "Training and validation evaluation horizons differ."
    )

if (
    train_token_cache["tokenizer_id"]
    != val_token_cache["tokenizer_id"]
):
    raise RuntimeError(
        "Training and validation tokenizer IDs differ."
    )

if (
    train_token_cache["tokenizer_revision"]
    != val_token_cache["tokenizer_revision"]
):
    raise RuntimeError(
        "Training and validation tokenizer revisions differ."
    )

print()
print(
    "Origin-aligned train/validation token-cache generation "
    "completed successfully."
)

print(
    "Training cache:",
    TRAIN_CACHE_PATH,
)

print(
    "Validation cache:",
    VAL_CACHE_PATH,
)


Project root: /Users/vishalruparelia/Desktop/Thesis/dynamic_graphs_thesis
Tokenizer loaded: NeoQuasar/Kronos-Tokenizer-base

Training sessions: 167
Validation sessions: 20
Training windows: 3173
Validation windows: 380
Dense horizons: [1, 2, 3, 4, 5] ... 60

Generating validation cache: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/origin_aligned_val_tokens.pt


Encoding origin-aligned token windows:   0%|          | 0/190 [00:00<?, ?it/s]


Saved validation cache to: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/origin_aligned_val_tokens.pt
Validation generation time: 1.14 minutes

Validation cache:
  context: (380, 60, 93, 2)
  target s1: (380, 60, 93)
  target s2: (380, 60, 93)
  evaluation truth: (380, 5, 93, 5)
  context mean: (380, 93, 6)
  context std: (380, 93, 6)
  target indices: (380, 60)

Validation future clipping by channel:
  open: 5.583899%
  high: 5.684022%
  low: 5.651669%
  close: 5.526929%
  volume: 1.765327%
  amount: 0.000000%

Generating training cache: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/origin_aligned_train_tokens.pt


Encoding origin-aligned token windows:   0%|          | 0/1587 [00:00<?, ?it/s]


Saved training cache to: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/origin_aligned_train_tokens.pt
Training generation time: 9.50 minutes

Training cache:
  context: (3173, 60, 93, 2)
  target s1: (3173, 60, 93)
  target s2: (3173, 60, 93)
  evaluation truth: (3173, 5, 93, 5)
  context mean: (3173, 93, 6)
  context std: (3173, 93, 6)
  target indices: (3173, 60)

Training future clipping by channel:
  open: 5.517629%
  high: 5.668601%
  low: 5.615170%
  close: 5.499098%
  volume: 1.700481%
  amount: 0.000000%

Origin-aligned train/validation token-cache generation completed successfully.
Training cache: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/origin_aligned_train_tokens.pt
Validation cache: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens/origin_aligned_v